# RLVR TerminalBench — Qwen2.5-7B on Colab T4

500-step GRPO training with Qwen2.5-7B-Instruct. Model cached on Google Drive.

**Runtime:** Go to Runtime > Change runtime type > Select **T4 GPU**

## 1. Mount Google Drive (caches model, survives restarts)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print(f'HuggingFace cache: {os.environ["HF_HOME"]}')

## 2. Install dependencies + clone repo

In [ ]:
!pip install -q transformers trl accelerate datasets peft bitsandbytes pyyaml tqdm

In [ ]:
# Clone the repo. AfterQuery is private, so store a GitHub PAT (repo scope) in
# Colab Secrets (key icon in the left sidebar) under the name GITHUB_TOKEN.
import os
import subprocess

REPO = 'AfterQuery'
REPO_URL = 'https://github.com/NathanG2022/AfterQuery.git'


def _scrub(text, token):
    return text.replace(token, '***') if token else text


token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if not os.path.isdir(REPO):
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    proc = subprocess.run(['git', 'clone', url], capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(
            'git clone failed:\n'
            + _scrub(proc.stderr, token)
            + '\nAfterQuery is private. Either add a GITHUB_TOKEN Colab secret, '
              'or upload the project as a zip instead:\n'
              '    from google.colab import files; files.upload()'
        )
    print('Cloned', REPO)
else:
    proc = subprocess.run(['git', '-C', REPO, 'pull'], capture_output=True, text=True)
    print(_scrub(proc.stdout + proc.stderr, token).strip())

os.chdir(REPO)
print('Working directory:', os.getcwd())

## 3. Verify GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU! Runtime > Change runtime type > T4 GPU')

## 4. Configuration

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Training params — tuned for T4 + 7B model
NUM_PROMPTS = 256
TOTAL_STEPS = 500
BATCH_SIZE = 4           # smaller batch to fit 7B in memory
NUM_GENERATIONS = 4      # more generations = better GRPO signal
LEARNING_RATE = 1e-5
MAX_NEW_TOKENS = 128
MAX_STEPS_PER_EPISODE = 3  # tighter pressure to solve quickly

# Output — saved to Google Drive so it persists
OUTPUT_DIR = '/content/drive/MyDrive/rlvr_models/qwen7b_500steps'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Model: {MODEL_NAME}')
print(f'Steps: {TOTAL_STEPS}, Batch: {BATCH_SIZE}, Generations: {NUM_GENERATIONS}')
print(f'Output: {OUTPUT_DIR}')

## 5. Build prompt dataset

In [ ]:
import sys
sys.path.insert(0, '.')

from datasets import Dataset
from envs.terminalbench_client import TerminalBenchClient
from envs.terminalbench_env import TerminalBenchEnv

tb_client = TerminalBenchClient(command_timeout=10, max_output_length=2000)
env = TerminalBenchEnv(
    tb_client,
    max_steps=MAX_STEPS_PER_EPISODE,
    step_penalty=0.01,
    w_success=1.0,
    w_eff=0.1,
    w_quality=0.05,
)

prompts = []
for _ in range(NUM_PROMPTS):
    obs = env.reset()
    prompts.append({'prompt': obs})

train_dataset = Dataset.from_list(prompts)
print(f'Dataset: {len(train_dataset)} prompts')
print(f'\nExample prompt:\n{prompts[0]["prompt"][:300]}...')

## 6. Define reward function

In [ ]:
# Reward function: shared implementation from the repo (envs/rewards.py).
#
# Two correctness-critical properties (previous inline version had neither):
#   1. Task matching — each completion is scored against the task its prompt
#      actually described, not a randomly re-sampled task.
#   2. Partial credit — graded verify() scores (0.3, 0.5, 0.7, ...) flow into
#      the reward instead of being collapsed to 0.
from envs.rewards import extract_command, make_reward_func

reward_func = make_reward_func(
    command_timeout=10,
    max_output_length=2000,
    step_penalty=0.01,
    w_success=1.0,
    w_eff=0.1,
    w_quality=0.05,
)
print('Reward function ready (task-matched, partial credit enabled)')

## 7. Load model (4-bit quantization + LoRA)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
)

print(f'Loading {MODEL_NAME} (first time downloads ~6GB to Google Drive)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
)

if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1e9
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU memory: {mem_used:.1f} / {mem_total:.1f} GB')

print('Model loaded!')

## 8. Train (500 steps GRPO)

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=NUM_GENERATIONS,
    num_train_epochs=1,
    max_steps=TOTAL_STEPS,
    max_completion_length=MAX_NEW_TOKENS,
    temperature=0.7,
    top_p=0.9,
    beta=0.04,
    logging_steps=10,
    save_strategy='steps',
    save_steps=100,
    report_to='none',
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_func,
    args=grpo_config,
    train_dataset=train_dataset,
    peft_config=peft_config,
)

print(f'Training {MODEL_NAME} for {TOTAL_STEPS} steps...')
print(f'Estimated time: ~1-2 hours on T4')
trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f'\nModel saved to {OUTPUT_DIR}')

## 9. Evaluate: trained vs base

In [ ]:
from transformers import AutoTokenizer
from statistics import mean

def evaluate_model(model_path, num_episodes=20, label='model'):
    """Evaluate a model on terminalbench tasks."""
    print(f'\nLoading {label}...')
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    eval_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
        ),
    ).eval()
    max_ctx = getattr(eval_model.config, 'max_position_embeddings', 2048)

    tb_client = TerminalBenchClient(command_timeout=10, max_output_length=2000)
    env = TerminalBenchEnv(
        tb_client, max_steps=MAX_STEPS_PER_EPISODE,
        step_penalty=0.01, w_success=1.0, w_eff=0.1, w_quality=0.05,
    )

    scores, steps = [], []
    for ep in range(num_episodes):
        obs = env.reset()
        done = False
        task_id = env.task.task_id if env.task else 'unknown'
        last_score = 0.0

        while not done:
            max_new = MAX_NEW_TOKENS
            inputs = tokenizer(
                obs, return_tensors='pt', truncation=True,
                max_length=max_ctx - max_new,
            ).to(eval_model.device)
            out_ids = eval_model.generate(
                inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                max_new_tokens=max_new,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
            new_ids = out_ids[0][inputs['input_ids'].shape[1]:]
            action = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
            action = extract_command(action)
            action = action.lstrip('$ ').strip()
            print(f'  ep{ep+1} step{env.step_count+1}: {action[:80]}')
            obs, _reward, done, info = env.step(action)
            last_score = info['success_score']

        scores.append(last_score)
        steps.append(info['step_count'])
        status = 'PASS' if last_score >= 1.0 else 'FAIL'
        print(f'  -> [{status}] task={task_id} score={last_score:.2f} steps={info["step_count"]}')

    del eval_model
    torch.cuda.empty_cache()

    return {
        'mean_score': mean(scores),
        'success_rate': mean(1.0 if s >= 1.0 else 0.0 for s in scores),
        'mean_steps': mean(steps),
    }

In [ ]:
NUM_EVAL_EPISODES = 30

print('=== Evaluating TRAINED model ===')
trained_results = evaluate_model(OUTPUT_DIR, num_episodes=NUM_EVAL_EPISODES, label='trained')

print(f'\n=== Evaluating BASE model ({MODEL_NAME}) ===')
base_results = evaluate_model(MODEL_NAME, num_episodes=NUM_EVAL_EPISODES, label='base')

print('\n' + '='*50)
print(f'{"Metric":<25} {"Base":>10} {"Trained":>10} {"Delta":>10}')
print('-'*50)
for key, label in [('mean_score', 'Mean score'), ('success_rate', 'Success rate'), ('mean_steps', 'Mean steps')]:
    b, t = base_results[key], trained_results[key]
    delta = t - b
    sign = '+' if delta >= 0 else ''
    print(f'{label:<25} {b:>10.3f} {t:>10.3f} {sign}{delta:>9.3f}')
print('='*50)

## 10. Save results summary

In [ ]:
# Save results to Google Drive
results_path = f'{OUTPUT_DIR}/eval_results.txt'
with open(results_path, 'w') as f:
    f.write(f'Model: {MODEL_NAME}\n')
    f.write(f'Training steps: {TOTAL_STEPS}\n')
    f.write(f'Batch size: {BATCH_SIZE}, Generations: {NUM_GENERATIONS}\n')
    f.write(f'Eval episodes: {NUM_EVAL_EPISODES}\n\n')
    f.write(f'{"Metric":<25} {"Base":>10} {"Trained":>10}\n')
    f.write('-'*50 + '\n')
    for key, label in [('mean_score', 'Mean score'), ('success_rate', 'Success rate'), ('mean_steps', 'Mean steps')]:
        f.write(f'{label:<25} {base_results[key]:>10.3f} {trained_results[key]:>10.3f}\n')

print(f'Results saved to {results_path}')
print(f'Model weights saved to {OUTPUT_DIR}')
print('\nAll saved to Google Drive — will persist across sessions!')